In [1]:
from pypdf import PdfReader
import os
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from sentence_transformers import CrossEncoder
import pickle
import faiss
import ollama
from rank_bm25 import BM25Okapi
import csv


/mnt/c/Users/Lenovo/OneDrive/Desktop/NOKIA/rag_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [113]:
documents = []
writer=csv.writer(open("extracted_text.csv", "w", newline=""))
for pdf_file in os.listdir("docs"):
    if pdf_file.endswith(".pdf"):

        reader = PdfReader(os.path.join("docs", pdf_file))

        for page_num, page in enumerate(reader.pages, start=1):

            extracted = page.extract_text()
            if extracted:

                documents.append({
                    "source": pdf_file,
                    "page": page_num,
                    "text": extracted
                })
for i in documents:
    writer.writerow([i["source"], i["page"], i["text"]])
print("Pages loaded:", len(documents))  

Pages loaded: 295


In [114]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = []

for doc in documents:
    text_chunks = splitter.split_text(doc["text"])

    for chunk in text_chunks:
        chunks.append({
        "source": doc["source"],
        "page": doc["page"],
        "text": chunk
    })

print("Total chunks:", len(chunks))
print(chunks[0]["source"])
print()
print(chunks[0]["text"])

Total chunks: 1631
nokia-annual-report-2025.pdf

Nokia in 2025


In [115]:
embedding_model = SentenceTransformer(
    "BAAI/bge-base-en-v1.5",
    device="cuda"
)
reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    device="cuda"
)

print("Reranker Loaded!")
print("Loaded!")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1134.81it/s]


Reranker Loaded!
Loaded!


In [116]:

tokenized_chunks = [
    chunk["text"].lower().split()
    for chunk in chunks
]


bm25 = BM25Okapi(tokenized_chunks)

print("BM25 index created!")

BM25 index created!


In [117]:
texts = [chunk["text"] for chunk in chunks]

embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print(embeddings.shape)

Batches:   0%|          | 0/51 [00:00<?, ?it/s]

Batches: 100%|██████████| 51/51 [00:16<00:00,  3.17it/s]

(1631, 768)


In [118]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(np.array(embeddings))
config = {
    "embedding_model": "BAAI/bge-base-en-v1.5",
    "reranker_model": "BAAI/bge-reranker-v2-m3",
    "chunk_size": 1000,
    "chunk_overlap": 200,
    "embedding_dimension": embeddings.shape[1]
}
print("Vectors stored:", index.ntotal)
faiss.write_index(index, "vector_db/vector_store.faiss")

with open("vector_db/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

with open("vector_db/config.pkl", "wb") as f:
    pickle.dump(config, f)

with open("vector_db/bm25.pkl", "wb") as f:
    pickle.dump(bm25, f)

print("Vectors stored:", index.ntotal)
print("FAISS index saved!")
print("Chunks saved!")
print("BM25 saved!")

Vectors stored: 1631
Vectors stored: 1631
FAISS index saved!
Chunks saved!
BM25 saved!


In [2]:
with open("vector_db/bm25.pkl", "rb") as f:
    bm25 = pickle.load(f)

with open("vector_db/config.pkl", "rb") as f:
    config = pickle.load(f)

index = faiss.read_index("vector_db/vector_store.faiss")

with open("vector_db/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

embedding_model = SentenceTransformer(
    config["embedding_model"],
    device="cuda"
)

reranker = CrossEncoder(
    config["reranker_model"],
    device="cuda"
)

print("Vector DB Loaded Successfully!")
print(f"Vectors: {index.ntotal}")
print(f"Chunks: {len(chunks)}")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1827.15it/s]


Vector DB Loaded Successfully!
Vectors: 1631
Chunks: 1631


In [3]:
print("FAISS Dimension:", index.d)
print("Embedding Dimension:", query_embedding.shape[1])

FAISS Dimension: 768


NameError: name 'query_embedding' is not defined

In [120]:
chat_history = []
def ask_rag(
    question,
    k=10,
    show_sources=True,
    show_retrieved_chunks=False
):

    # Create query embedding
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    distances, indices = index.search(query_embedding, k)
    # BM25 Search
    tokenized_query = question.lower().split()

    bm25_scores = bm25.get_scores(tokenized_query)

    bm25_indices = np.argsort(bm25_scores)[::-1][:k]
    # Merge FAISS + BM25 results
    candidate_indices = list(indices[0])
    pairs = [
    (question, chunks[idx]["text"])
    for idx in candidate_indices
    ]
    for idx in bm25_indices:
        if idx not in candidate_indices:
            candidate_indices.append(idx)

    print(f"Candidates after merge: {len(candidate_indices)}")

    scores = reranker.predict(pairs)

    ranked_results = sorted(
        zip(scores, candidate_indices), 
        key=lambda x: x[0],
        reverse=True
    )
    print("\nTop Retrieved Chunks After Reranking:\n")
    for score, idx in ranked_results:
        print(f"{score:.4f} -> {chunks[idx]['source']} Page {chunks[idx]['page']}")
    
    top_chunks = ranked_results[:5]

    context_parts = []
    sources = []

    for rank, (score, idx) in enumerate(top_chunks, start=1):

        chunk = chunks[idx]

        context_parts.append(
            f"""
SOURCE: {chunk['source']}
PAGE: {chunk['page']}
RERANK SCORE: {score:.4f}

{chunk['text']}
"""
        )

        sources.append(
            f"{chunk['source']} (Page {chunk['page']})"
        )

    context = "\n\n".join(context_parts)
    history = ""

    for item in chat_history[-5:]:    
        history += f"""
    User: {item['question']}
    Assistant: {item['answer']}
    """
    prompt = f"""
You are a document question-answering assistant.

STRICT RULES:

1. Answer ONLY using information found in the provided context.
2. Do NOT use outside knowledge.
3. If the answer is not present in the context, respond exactly:
"I could not find that information in the documents."
4. Combine information from multiple chunks if necessary.
5. Never invent facts.
6. Cite only the provided documents.
7. Use conversation history to resolve references like(these are just examples and not limited to these words):
   - it
   - they
   - that product
   - this feature
   
Conversation History:
{history}

Document Context:
{context}

Current Question:
{question}

ANSWER:
"""

    response = ollama.chat(
        model="qwen3:8b",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    answer = response["message"]["content"]
    chat_history.append(
        {
            "question": question,
            "answer": answer
        }
    )
    result = {
        "answer": answer,
        "sources": sorted(set(sources))
    }

    if show_retrieved_chunks:
        result["retrieved_chunks"] = [
            chunks[idx]["text"][:500]
            for score, idx in top_chunks
        ]

    return result

In [1]:
result = ask_rag(
    "what are Nokia’s four strategic impact areas also explain these impact areas"
)   

print(result["answer"])

print("\nSources:")
for src in result["sources"]:
    print("-", src)

NameError: name 'ask_rag' is not defined

In [122]:
result = ask_rag(
    "Explain these furthur"
)

print(result["answer"])

Candidates after merge: 20

Top Retrieved Chunks After Reranking:

0.0001 -> nokia-annual-report-2025.pdf Page 159
0.0001 -> nokia-annual-report-2025.pdf Page 246
0.0001 -> nokia-annual-report-2025.pdf Page 217
0.0000 -> nokia-annual-report-2025.pdf Page 245
0.0000 -> nokia-annual-report-2025.pdf Page 249
0.0000 -> nokia-annual-report-2025.pdf Page 257
0.0000 -> nokia-annual-report-2025.pdf Page 253
0.0000 -> nokia-annual-report-2025.pdf Page 253
0.0000 -> nokia-annual-report-2025.pdf Page 276
0.0000 -> nokia-annual-report-2025.pdf Page 202
I could not find that information in the documents.


In [97]:
result=ask_rag(
    "Give me all the details of balarang public school"
)
print(result["answer"])

Candidates after merge: 18

Top Retrieved Chunks After Reranking:

0.6959 -> List_of_government_schools_in_New_South_Wales_(A–C).pdf Page 21
0.6316 -> List_of_government_schools_in_New_South_Wales_(A–C).pdf Page 21
0.5066 -> List_of_government_schools_in_New_South_Wales_(A–C).pdf Page 4
0.0087 -> List_of_government_schools_in_New_South_Wales_(A–C).pdf Page 23
0.0043 -> List_of_government_schools_in_New_South_Wales_(A–C).pdf Page 22
0.0031 -> List_of_government_schools_in_New_South_Wales_(A–C).pdf Page 36
0.0030 -> List_of_government_schools_in_New_South_Wales_(A–C).pdf Page 34
0.0026 -> List_of_government_schools_in_New_South_Wales_(A–C).pdf Page 49
0.0011 -> List_of_government_schools_in_New_South_Wales_(A–C).pdf Page 48
0.0001 -> List_of_government_schools_in_New_South_Wales_(A–C).pdf Page 54
Balarang Public School is located in **Oak Flats**, part of the **Illawarra** region. It was opened in **1968** and has the following coordinates: **34°33′33.22″S 150°49′50.69″E**.  

Citations:

In [98]:
result=ask_rag(
    "Number of woman serving as the head of state?"
)
print(result["answer"])

Candidates after merge: 17

Top Retrieved Chunks After Reranking:

0.9949 -> 2020s.pdf Page 93
0.0449 -> 2020s.pdf Page 93
0.0206 -> 2020s.pdf Page 135
0.0159 -> 2020s.pdf Page 54
0.0134 -> 2020s.pdf Page 55
0.0099 -> 2020s.pdf Page 171
0.0082 -> 2020s.pdf Page 105
0.0061 -> 2020s.pdf Page 143
0.0048 -> 2020s.pdf Page 59
0.0037 -> 2020s.pdf Page 16
The number of women serving as head of state, as stated in the documents, was **11** as of **June 2019**.  

Citations:  
- [889] (from *2020s.pdf*, page 93)  

Note: The documents mention additional female heads of state elected after 2019 (e.g., Claudia Sheinbaum in 2024), but no updated total is provided.


In [74]:
result=ask_rag(
    "So what is the difference between them"
)
print(result["answer"])

Candidates after merge: 19

Top Retrieved Chunks After Reranking:

0.1992 -> doc2.pdf Page 139
0.0632 -> doc1.pdf Page 99
0.0301 -> doc2.pdf Page 29
0.0169 -> doc2.pdf Page 110
0.0162 -> doc2.pdf Page 112
0.0045 -> doc2.pdf Page 119
0.0038 -> doc1.pdf Page 41
0.0035 -> doc2.pdf Page 140
0.0033 -> doc2.pdf Page 141
0.0006 -> doc1.pdf Page 2
The difference between CWDM (Coarse Wavelength Division Multiplexing) and DWDM (Dense Wavelength Division Multiplexing) lies in their **channel spacings**, **DFB lasers**, and **transmission distances**:  

1. **Channel Spacings**:  
   - CWDM systems have a channel spacing of **20 nm**.  
   - DWDM systems use much narrower spacing, typically **0.8 nm (100 GHz)**, as defined by the ITU standard.  

2. **DFB Lasers**:  
   - DWDM systems predominantly use **DFB (Distributed Feedback) lasers**, which provide precise wavelength control required for dense spacing.  
   - CWDM systems generally do not rely on DFB lasers.  

3. **Transmission Distances**:

In [76]:
import time
import numpy as np
import ollama

# ----------------------------
# Test Question
# ----------------------------
question = "What are Directional Couplers?"
k = 10

print("=" * 60)
print("RAG PROFILING")
print("=" * 60)

# ----------------------------
# Embedding
# ----------------------------
t0 = time.perf_counter()

query_embedding = embedding_model.encode(
    [question],
    convert_to_numpy=True,
    normalize_embeddings=True
)

embedding_time = time.perf_counter() - t0

# ----------------------------
# FAISS
# ----------------------------
t1 = time.perf_counter()

distances, indices = index.search(query_embedding, k)

faiss_time = time.perf_counter() - t1

# ----------------------------
# BM25
# ----------------------------
t2 = time.perf_counter()

tokenized_query = question.lower().split()

bm25_scores = bm25.get_scores(tokenized_query)

bm25_indices = np.argsort(bm25_scores)[::-1][:k]

bm25_time = time.perf_counter() - t2

# ----------------------------
# Merge
# ----------------------------
t3 = time.perf_counter()

candidate_indices = list(indices[0])

for idx in bm25_indices:
    if idx not in candidate_indices:
        candidate_indices.append(idx)

merge_time = time.perf_counter() - t3

# ----------------------------
# Reranker
# ----------------------------
pairs = [
    (question, chunks[idx]["text"])
    for idx in candidate_indices
]

t4 = time.perf_counter()

scores = reranker.predict(pairs)

reranker_time = time.perf_counter() - t4

# ----------------------------
# Prompt
# ----------------------------
ranked_results = sorted(
    zip(scores, candidate_indices),
    key=lambda x: x[0],
    reverse=True
)

top_chunks = ranked_results[:3]

context = "\n\n".join(
    chunks[idx]["text"]
    for score, idx in top_chunks
)

prompt = f"""
Context:
{context}

Question:
{question}

Answer:
"""

# ----------------------------
# Ollama
# ----------------------------
t5 = time.perf_counter()

response = ollama.chat(
    model="qwen3:8b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

llm_time = time.perf_counter() - t5

# ----------------------------
# Results
# ----------------------------
total = (
    embedding_time +
    faiss_time +
    bm25_time +
    merge_time +
    reranker_time +
    llm_time
)

print(f"Embedding : {embedding_time:.4f} sec")
print(f"FAISS     : {faiss_time:.4f} sec")
print(f"BM25      : {bm25_time:.4f} sec")
print(f"Merge     : {merge_time:.4f} sec")
print(f"Reranker  : {reranker_time:.4f} sec")
print(f"LLM       : {llm_time:.4f} sec")
print("-" * 60)
print(f"TOTAL     : {total:.4f} sec")
print("=" * 60)

print("\nAnswer Preview:\n")
print(response["message"]["content"][:500])

RAG PROFILING
Embedding : 0.0513 sec
FAISS     : 0.0022 sec
BM25      : 0.0013 sec
Merge     : 0.0001 sec
Reranker  : 0.6182 sec
LLM       : 106.4627 sec
------------------------------------------------------------
TOTAL     : 107.1358 sec

Answer Preview:

**Directional Couplers** are passive microwave devices that sample a small portion of microwave power from a main waveguide to a secondary supporting waveguide. They are 4-port waveguide junctions designed to couple power unidirectionally or bidirectionally, depending on the direction of signal flow. Here's a structured explanation:

### **Key Characteristics:**
1. **Structure:**  
   - Composed of a **primary main waveguide** and a **secondary supporting waveguide**.  
   - Physical structures 


In [ ]:
import gc
import torch

del embedding_model
del reranker

gc.collect()
torch.cuda.empty_cache()

print(torch.cuda.memory_allocated()/1024**3)
print(torch.cuda.memory_reserved()/1024**3)

0.0079345703125
0.01953125
